# CALM — Permutation-Invariant Calibrated Readout (standalone)
Training-free, test-time. Loads your trained 7B adapter (auto-found in /kaggle/input), then compares base vs PriDe vs cyclic-marginalized readouts with bootstrap CIs. No token generation; ~50–90 min on T4. Artifacts -> `calm_pi_artifacts.zip`.

**Only input needed:** your trained adapter (already mounted). Set `ADAPTER_SRC` in CONFIG if auto-detect misses it.


In [ ]:
# === Secrets (optional, NO hardcoded keys) ===
import os
def _load_secret(name):
    v = os.environ.get(name)
    if v: return v
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: os.environ[name] = v; return v
    except Exception:
        pass
    return None
HF_TOKEN = _load_secret("HF_TOKEN")
print("HF_TOKEN:", bool(HF_TOKEN), "(optional)")

# === Installs ===
import sys, subprocess
def pip(*p): subprocess.check_call([sys.executable,"-m","pip","install","-q",*p])
def ensure(mod, pkg=None):
    try: __import__(mod)
    except Exception: pip(pkg or mod)
for mod, pkg in [("datasets","datasets"),("transformers","transformers>=4.41"),
                 ("accelerate","accelerate"),("sklearn","scikit-learn"),
                 ("matplotlib","matplotlib"),("bitsandbytes","bitsandbytes"),
                 ("peft","peft"),("huggingface_hub","huggingface_hub")]:
    ensure(mod,pkg)
print("installs ok")




In [ ]:
# === Imports & devices ===
import os, gc, json, csv, math, random, re, time, glob, urllib.request, warnings
from dataclasses import dataclass
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM","false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import numpy as np, torch
import torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset, Dataset
assert torch.cuda.is_available(), "CUDA required."
N_GPU = torch.cuda.device_count()
STUDENT_DEVICE = torch.device("cuda:0")
OPTIONS = list("ABCD")
print("GPUs=", N_GPU, "|", torch.cuda.get_device_name(0))




In [ ]:
# ============================= CONFIG =============================
# Point ADAPTER_SRC at your trained 7B LoRA adapter. Three ways (auto-detected):
#   1) leave "AUTO" and add the adapter folder as a Kaggle Input (it finds adapter_config.json)
#   2) a HF Hub id, e.g. "sabarna07/your-medqa-7b-adapter"
#   3) a local path, e.g. "/kaggle/input/medqa-7b-adapter/best_student_adapter"
ADAPTER_SRC  = "AUTO"
BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ATTACH_EXISTING = True   # if a `student` already lives in this kernel, reuse it

# ---- CALM-R3 run knobs ----
REASON_TEST_N = 600      # test items to reason over (set 1273 for the full split)
RUN_PHASE1    = False    # set True ONLY after Phase 0 prints GO

@dataclass
class CFG:
    seed: int = 2000
    val_n: int = 1000
    test_n: int = 1273
    seed_labeled_n: int = 2000
    unlabeled_n: int = 4000
    eval_bs: int = 8
    max_src_len: int = 448
    ece_bins: int = 10
    reason_cot_tokens: int = 256
    grad_clip: float = 1.0
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_targets: tuple = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")
    dataset_candidates: tuple = ("sabarna07/MedQA-USMLE-4-options-hf","GBaker/MedQA-USMLE-4-options-hf")
    local_data_dir: str = ""
    time_budget_hours: float = 11.0
    out_dir: str = "./paper_artifacts_v3_qwen7b"
cfg = CFG()
os.makedirs(cfg.out_dir, exist_ok=True)
START_TIME = time.time(); DEADLINE = START_TIME + cfg.time_budget_hours*3600
def time_left(): return DEADLINE - time.time()
def have_time(need=0.0): return time_left() > need
print("config ready | out_dir:", cfg.out_dir)




In [ ]:
# ============================= DATA (same loader & splits as the main run) =============================
def _get_options(ex):
    o = ex.get("options") or ex.get("choices") or ex.get("endings")
    if isinstance(o, dict):
        return [str(o.get(k, o.get(k.lower(), ""))).strip() for k in OPTIONS]
    if isinstance(o, list) and len(o) >= 4:
        if isinstance(o[0], dict) and "value" in o[0]:
            o = [d["value"] for d in sorted(o, key=lambda x: x.get("key",""))]
        return [str(x).strip() for x in o[:4]]
    for ks in [("opa","opb","opc","opd"), ("ending0","ending1","ending2","ending3")]:
        if all(k in ex for k in ks): return [str(ex[k]).strip() for k in ks]
    return None
def normalize_example(ex):
    q = ex.get("sent1") or ex.get("question") or ex.get("stem") or ex.get("prompt") or ""
    if ex.get("sent2") and str(ex["sent2"]).strip() not in ("", "?"):
        q = (str(q).strip()+"\n"+str(ex["sent2"]).strip()).strip()
    opts = _get_options(ex) or ["","","",""]
    lab = ex.get("answer_idx", ex.get("label", ex.get("answer", None)))
    ans = None
    if isinstance(lab, int): ans = OPTIONS[lab] if 0<=lab<4 else None
    elif lab is not None and str(lab).strip().isdigit(): ans = OPTIONS[int(str(lab).strip())]
    elif isinstance(lab, str) and lab.strip().upper() in OPTIONS: ans = lab.strip().upper()
    elif isinstance(lab, str):
        for i,o in enumerate(opts):
            if o and o.strip().lower() == lab.strip().lower(): ans = OPTIONS[i]; break
    if ans is None: ans = "A"
    return {"question": str(q).strip(), "options": opts, "answer": ans}
def normalize_split(split, max_n=None):
    n = len(split) if max_n is None else min(len(split), max_n)
    return Dataset.from_list([normalize_example(split[i]) for i in range(n)])

def _read_records(text):
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list): return obj
        if isinstance(obj, dict):
            for v in obj.values():
                if isinstance(v, list): return v
            return [obj]
    except Exception: pass
    rows=[]
    for line in text.splitlines():
        line=line.strip()
        if not line: continue
        try: rows.append(json.loads(line))
        except Exception: pass
    return rows
def _http_get(url, token=None, timeout=180):
    h={"User-Agent":"Mozilla/5.0"}
    if token: h["Authorization"]="Bearer "+token
    with urllib.request.urlopen(urllib.request.Request(url, headers=h), timeout=timeout) as r:
        return r.read().decode("utf-8")
HF_DIRECT = [("sabarna07/MedQA-USMLE-4-options-hf", {"train":"train.json","validation":"dev.json","test":"test.json"}),
             ("GBaker/MedQA-USMLE-4-options-hf", {"train":"train.json","validation":"dev.json","test":"test.json"})]
raw=None; used_id=None
for repo, fnames in HF_DIRECT:
    try:
        base="https://huggingface.co/datasets/%s/resolve/main/"%repo; d={}
        for split, fn in fnames.items():
            recs=_read_records(_http_get(base+fn, token=HF_TOKEN))
            if not recs: raise ValueError("empty "+fn)
            d[split]=Dataset.from_list(recs)
        raw=d; used_id=repo+" (direct http)"
        print("Loaded:", repo, {k:len(v) for k,v in d.items()}); break
    except Exception as e:
        print("[http skip]", repo, type(e).__name__, str(e)[:80])
if raw is None:
    for cand in cfg.dataset_candidates:
        try:
            raw=load_dataset(cand, token=HF_TOKEN); used_id=cand
            print("Loaded via load_dataset:", cand, list(raw.keys())); break
        except Exception as e: print("[resolver skip]", cand, type(e).__name__)
assert raw is not None, "Dataset load failed: add the MedQA dataset as a Kaggle Input or check network."
sp=dict(raw)
if "validation" not in sp:
    tt=sp["train"].train_test_split(test_size=0.1, seed=cfg.seed); sp["train"],sp["validation"]=tt["train"],tt["test"]
if "test" not in sp: sp["test"]=sp["validation"]
full_train=normalize_split(sp["train"])
val_ds  = normalize_split(sp["validation"], max_n=cfg.val_n)
test_ds = normalize_split(sp["test"], max_n=cfg.test_n)
idx=list(range(len(full_train))); random.Random(cfg.seed).shuffle(idx)
seed_ds=Dataset.from_list([full_train[i] for i in idx[:cfg.seed_labeled_n]])
print("seed=%d val=%d test=%d"%(len(seed_ds),len(val_ds),len(test_ds)))




In [ ]:
# ============================= MODEL + ADAPTER + HELPERS =============================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

def _resolve_adapter(src):
    if src and src not in ("AUTO", "") and os.path.isdir(src):
        return src
    roots = ["/kaggle/input", "/kaggle/working", cfg.out_dir, "."]
    hits = []
    for r in roots:
        hits += glob.glob(os.path.join(r, "**", "adapter_config.json"), recursive=True)
    dirs = sorted(set(os.path.dirname(h) for h in hits))
    if dirs:
        pref   = [d for d in dirs if "best_student" in d.lower()]
        nonref = [d for d in dirs if "reflect" not in d.lower()]
        pick = (pref or nonref or dirs)[0]
        print("adapter candidates:", dirs)
        print("SELECTED adapter:", pick, "  <-- verify this is best_student_adapter!")
        return pick
    if src and src not in ("AUTO", ""):
        print("no local adapter found; trying as HF Hub id:", src)
        return src
    raise FileNotFoundError("No adapter found. Set ADAPTER_SRC to the best_student_adapter path or a HF Hub id.")

if ATTACH_EXISTING and ("student" in globals()) and ("student_tok" in globals()):
    print("Reusing existing `student` from this kernel.")
else:
    ADAPTER = _resolve_adapter(ADAPTER_SRC); print("Loading adapter:", ADAPTER)
    student_tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, token=HF_TOKEN)
    student_tok.padding_side="left"; student_tok.truncation_side="left"
    if student_tok.pad_token is None: student_tok.pad_token = student_tok.eos_token
    _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    _base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=_bnb,
                device_map={"": str(STUDENT_DEVICE)}, low_cpu_mem_usage=True, token=HF_TOKEN)
    _base.config.use_cache=False
    _base = prepare_model_for_kbit_training(_base, use_gradient_checkpointing=True)
    student = PeftModel.from_pretrained(_base, ADAPTER, is_trainable=True, token=HF_TOKEN)
    print("student ready:", BASE_MODEL, "+ LoRA")
student.eval()

def format_prompt(q, opts):
    return ("You are taking a USMLE-style multiple-choice exam.\n"
            "Choose the single best answer.\n\nQuestion: " + q + "\n\nOptions:\n"
            "A. " + opts[0] + "\nB. " + opts[1] + "\nC. " + opts[2] + "\nD. " + opts[3] + "\n\n"
            "Reply with ONLY one letter: A, B, C, or D.")
def stu_chat(user, system="You are a medical expert. Answer with a single letter."):
    if hasattr(student_tok, "apply_chat_template"):
        return student_tok.apply_chat_template(
            [{"role":"system","content":system},{"role":"user","content":user}],
            tokenize=False, add_generation_prompt=True)
    return system + "\n\n" + user
def _build_letter_ids(tok):
    out={}
    for L in OPTIONS:
        c=[]
        for v in [L, " "+L]:
            e=tok.encode(v, add_special_tokens=False)
            if len(e)==1: c.append(e[0])
        out[L]=c or [tok.encode(L, add_special_tokens=False)[0]]
    return out
STU_LETTER_IDS=_build_letter_ids(student_tok); print("letter ids:", STU_LETTER_IDS)
def _fwd_last(model, input_ids, attention_mask):
    for kw in ("logits_to_keep","num_logits_to_keep"):
        try: return model(input_ids=input_ids, attention_mask=attention_mask, **{kw:1}).logits[:, -1, :]
        except TypeError: continue
    return model(input_ids=input_ids, attention_mask=attention_mask).logits[:, -1, :]
def _letter_logits(last):
    cols=[torch.logsumexp(torch.stack([last[:, t] for t in STU_LETTER_IDS[L]], 0), 0) for L in OPTIONS]
    return torch.stack(cols, 1)
@torch.no_grad()
def student_option_logits(model, prompts, bs=None):
    bs=bs or cfg.eval_bs; model.eval(); out=[]
    for i in range(0, len(prompts), bs):
        chats=[stu_chat(p) for p in prompts[i:i+bs]]
        enc=student_tok(chats, return_tensors="pt", padding=True, truncation=True,
                        max_length=cfg.max_src_len).to(STUDENT_DEVICE)
        out.append(_letter_logits(_fwd_last(model, enc.input_ids, enc.attention_mask)).float().cpu()); del enc
    torch.cuda.empty_cache(); return torch.cat(out, 0)
def expected_calibration_error(correct, conf, n_bins=10):
    correct=np.asarray(correct,float); conf=np.asarray(conf,float)
    bins=np.linspace(0,1,n_bins+1); e=0.0; n=len(correct)
    for i in range(n_bins):
        lo,hi=bins[i],bins[i+1]
        m=(conf>lo)&(conf<=hi) if i>0 else (conf>=lo)&(conf<=hi)
        if m.sum()==0: continue
        e+=(m.sum()/n)*abs(correct[m].mean()-conf[m].mean())
    return float(e)
def fit_temperature(logits, labels, max_iter=200):
    logits=torch.tensor(np.asarray(logits),dtype=torch.float32); labels=torch.tensor(np.asarray(labels),dtype=torch.long)
    logT=torch.zeros(1, requires_grad=True); opt=torch.optim.LBFGS([logT], lr=0.1, max_iter=max_iter)
    def closure():
        opt.zero_grad(); loss=F.cross_entropy(logits/logT.exp(), labels); loss.backward(); return loss
    opt.step(closure); return float(logT.exp().item())
def _parse_letter(t):
    if not t: return None
    pats=[r"\\boxed\{\s*([ABCD])\b", r"final\s+answer[^A-Za-z]{0,20}([ABCD])\b",
          r"correct\s+(?:answer|option|choice)\s*(?:is|:|=)?\s*\(?\s*([ABCD])\b",
          r"\banswer\s*(?:is|:|=)\s*\(?\s*([ABCD])\b", r"(?:^|\n)\s*\(?([ABCD])\)?\s*[.)]?\s*$"]
    for p in pats:
        ms=list(re.finditer(p, t, re.I))
        if ms: return ms[-1].group(1).upper()
    return None
print("helpers ready")





## Permutation-invariant experiment


In [ ]:
# =====================================================================================
# CALM permutation-invariant calibrated readout  (training-free, test-time)
#   Compares on the trained 7B student, with bootstrap CIs (honest protocol):
#     (a) BASE     : canonical single-pass calibrated readout            (1 forward/item)
#     (b) PriDe    : option-ID prior debiasing                           (~1 forward/item)
#     (c) CYC-MARG : 4-way cyclic-permutation marginalization (ours)     (4 forward/item)
#   Reports acc (+CI), ECE, AURC, cov@80, per-shift accuracy spread (robustness),
#   and paired CIs of (method - base). No token generation -> fast & cheap.
# Requires infra: student, student_tok, OPTIONS, cfg, val_ds, test_ds, format_prompt,
#   student_option_logits, expected_calibration_error, fit_temperature.
# =====================================================================================
import os, json, csv, time, numpy as np, torch, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

PI = os.path.join(cfg.out_dir, "calm_pi"); os.makedirs(PI, exist_ok=True)
SHIFTS    = (0, 1, 2, 3)
VAL_FIT_N = 500     # val items for temperature + PriDe prior (None = all 1000)
BOOT_N    = 2000


def _sm(z, T=1.0):
    z = np.asarray(z, float) / T; z = z - z.max(-1, keepdims=True)
    e = np.exp(z); return e / e.sum(-1, keepdims=True)

def content_probs_over_shifts(ds, shifts=SHIFTS):
    """Evaluate all cyclic shifts; read per-LETTER softmax back to per-CONTENT prob.
    Under shift s, displayed letter j shows original content (j+s)%4, so content c
    sits at letter (c-s)%4. Returns P_marg[N,4], golds[N], base_logits[N,4](s=0),
    letter_probs[S,N,4]."""
    N = len(ds); prompts = []
    for s in shifts:
        for i in range(N):
            opts = ds[i]["options"]; disp = [opts[(j + s) % 4] for j in range(4)]
            prompts.append(format_prompt(ds[i]["question"], disp))
    t0 = time.time()
    logits = student_option_logits(student, prompts).numpy().reshape(len(shifts), N, 4)
    print("    %d forward prompts in %.1f min" % (len(prompts), (time.time() - t0) / 60))
    letter_probs = _sm(logits)
    content = np.zeros_like(letter_probs)
    for si, s in enumerate(shifts):
        for c in range(4):
            content[si, :, c] = letter_probs[si, :, (c - s) % 4]
    P = content.mean(0)
    golds = np.array([OPTIONS.index(ds[i]["answer"]) for i in range(N)])
    return P, golds, logits[0], letter_probs

def _trapz(y, x):
    fn = getattr(np, "trapezoid", None) or getattr(np, "trapz", None)
    return float(fn(y, x))
def aurc(correct, conf):
    o = np.argsort(-conf); c = np.asarray(correct)[o]
    return _trapz(1 - np.cumsum(c) / np.arange(1, len(c) + 1), np.arange(1, len(c) + 1) / len(c))
def cov_at_acc(correct, conf, tgt=0.80):
    o = np.argsort(-conf); c = np.asarray(correct)[o]; run = np.cumsum(c) / np.arange(1, len(c) + 1)
    ok = np.where(run >= tgt)[0]; return float((ok.max() + 1) / len(c)) if len(ok) else 0.0
def boot_acc(correct, n=BOOT_N, seed=1):
    c = np.asarray(correct, float); N = len(c); rng = np.random.default_rng(seed)
    a = np.array([c[rng.integers(0, N, N)].mean() for _ in range(n)])
    return float(c.mean()), (float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))
def boot_diff(a, b, n=BOOT_N, seed=0):
    a = np.asarray(a, float); b = np.asarray(b, float); N = len(a); rng = np.random.default_rng(seed)
    d = np.array([(a[s].mean() - b[s].mean()) for s in (rng.integers(0, N, N) for _ in range(n))])
    return float(a.mean() - b.mean()), (float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))

def report(name, P, golds):
    pred = P.argmax(1); conf = P.max(1); correct = (pred == golds).astype(int)
    acc, ci = boot_acc(correct)
    ece = expected_calibration_error(correct, conf, cfg.ece_bins)
    oh = np.eye(4)[golds]; brier = float(np.mean(np.sum((P - oh) ** 2, 1)))
    a = aurc(correct, conf); cov = cov_at_acc(correct, conf, 0.80)
    print("[%-16s] acc=%.4f CI[%.4f,%.4f]  ECE=%.4f  Brier=%.3f  AURC=%.4f  cov@80=%.3f"
          % (name, acc, ci[0], ci[1], ece, brier, a, cov))
    return dict(acc=acc, acc_ci=ci, ece=ece, brier=brier, aurc=a, cov80=cov), correct, conf


print("\n" + "=" * 72 + "\nCALM permutation-invariant calibrated readout\n" + "=" * 72)
student.eval()
valfit = [val_ds[i] for i in range(min(VAL_FIT_N or len(val_ds), len(val_ds)))]
print(">> validation shifts (for temperature + prior) ...")
Pv, gv, baseLv, lpv = content_probs_over_shifts(valfit)
print(">> test shifts (full split) ...")
Pt, gt, baseLt, lpt = content_probs_over_shifts(test_ds)

print("\n--- TEST (n=%d) ---" % len(gt))
# (a) BASE calibrated
Tb = fit_temperature(baseLv, gv)
r_base, c_base, _ = report("BASE cal (T=%.2f)" % Tb, _sm(baseLt, Tb), gt)
# (b) PriDe
prior = lpv.reshape(-1, 4).mean(0); prior = prior / prior.sum()
canon_v = _sm(baseLv); canon_t = _sm(baseLt)
prideLv = np.log(np.clip(canon_v / prior[None, :], 1e-9, 1)); prideLt = np.log(np.clip(canon_t / prior[None, :], 1e-9, 1))
Tp = fit_temperature(prideLv, gv)
r_pride, c_pride, _ = report("PriDe cal (T=%.2f)" % Tp, _sm(prideLt, Tp), gt)
print("    option-ID prior A/B/C/D =", np.round(prior, 3).tolist())
# (c) CYC-MARG (ours)
cycLv = np.log(np.clip(Pv, 1e-9, 1)); cycLt = np.log(np.clip(Pt, 1e-9, 1))
Tc = fit_temperature(cycLv, gv)
r_cyc, c_cyc, conf_cyc = report("CYC-MARG (T=%.2f)" % Tc, _sm(cycLt, Tc), gt)

# paired gains vs base
d_pride, ci_pride = boot_diff(c_pride, c_base)
d_cyc, ci_cyc = boot_diff(c_cyc, c_base)
print("\nPriDe   - base = %+.4f  CI[%+.4f,%+.4f]  -> %s" % (d_pride, *ci_pride, "significant" if ci_pride[0] > 0 else "n.s."))
print("CYC-MARG- base = %+.4f  CI[%+.4f,%+.4f]  -> %s" % (d_cyc, *ci_cyc, "significant" if ci_cyc[0] > 0 else "n.s."))

# robustness: per-shift base accuracy spread (cyc-marg is invariant by construction)
per_shift = []
for si in range(lpt.shape[0]):
    pc = np.zeros((len(gt), 4))
    for c in range(4): pc[:, c] = lpt[si, :, (c - SHIFTS[si]) % 4]
    per_shift.append(float((pc.argmax(1) == gt).mean()))
print("\nper-shift BASE acc:", [f"{a:.3f}" for a in per_shift],
      "  spread=%.3f  std=%.4f" % (max(per_shift) - min(per_shift), float(np.std(per_shift))))
print("CYC-MARG is permutation-invariant by construction (per-question prediction is order-independent).")

# save
out = {"n_test": len(gt), "T_base": Tb, "T_pride": Tp, "T_cyc": Tc,
       "base": r_base, "pride": r_pride, "cyc_marg": r_cyc,
       "pride_minus_base": d_pride, "pride_ci": ci_pride,
       "cyc_minus_base": d_cyc, "cyc_ci": ci_cyc,
       "id_prior_ABCD": prior.tolist(), "per_shift_base_acc": per_shift,
       "per_shift_spread": float(max(per_shift) - min(per_shift)),
       "per_shift_std": float(np.std(per_shift))}
def _ser(o):
    if isinstance(o, dict): return {k: _ser(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_ser(v) for v in o]
    return o
json.dump(_ser(out), open(os.path.join(PI, "pi_results.json"), "w"), indent=2)

# plots
plt.figure(figsize=(5, 3.4))
names = ["base", "PriDe", "cyc-marg"]; accs = [r_base["acc"], r_pride["acc"], r_cyc["acc"]]
errs = [[a - lo for a, (lo, hi) in [(r_base["acc"], r_base["acc_ci"]), (r_pride["acc"], r_pride["acc_ci"]), (r_cyc["acc"], r_cyc["acc_ci"])]],
        [hi - a for a, (lo, hi) in [(r_base["acc"], r_base["acc_ci"]), (r_pride["acc"], r_pride["acc_ci"]), (r_cyc["acc"], r_cyc["acc_ci"])]]]
plt.bar(names, accs, yerr=errs, capsize=5, color=["#888", "#5a9", "#36c"])
plt.ylim(min(accs) - 0.03, max(accs) + 0.03); plt.ylabel("test accuracy"); plt.title("Readout accuracy (95% CI)")
plt.tight_layout(); plt.savefig(os.path.join(PI, "pi_accuracy.png"), dpi=200); plt.close()

def _rc(correct, conf):
    o = np.argsort(-conf); c = np.asarray(correct)[o]
    return np.arange(1, len(c) + 1) / len(c), 1 - np.cumsum(c) / np.arange(1, len(c) + 1)
plt.figure(figsize=(5, 3.4))
cb, rb = _rc(c_base, _sm(baseLt, Tb).max(1)); cc, rc = _rc(c_cyc, conf_cyc)
plt.plot(cb, rb, label="base"); plt.plot(cc, rc, label="cyc-marg (ours)")
plt.xlabel("coverage"); plt.ylabel("risk (error)"); plt.title("Risk–coverage"); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(PI, "pi_risk_coverage.png"), dpi=200); plt.close()

import shutil
zb = os.path.join(cfg.out_dir, "calm_pi_artifacts")
shutil.make_archive(zb, "zip", PI)
print("\nzipped ->", zb + ".zip", "| contents:", sorted(os.listdir(PI)))
print("\nSEND ME: pi_results.json  (or the BASE / PriDe / CYC-MARG lines + the two paired CIs).")


